# M3L3 E07 — Anatomía de un grafo LangGraph (Resolution)
### Módulo 3 · Lecture 3 · Sistemas Multiagente

## ¿Qué vas a aprender hoy?
- identificar las cinco piezas de un grafo LangGraph.
- definir un State con `TypedDict`.
- registrar nodos y conectarlos con edges.
- compilar e invocar el primer grafo con un LLM real.


## ¿Qué necesitás saber antes?

Venís de E00–E06 donde construiste orquestadores, routers y protocolos de output en Python puro. Ahora llegamos a LangGraph: el mismo patrón arquitectónico, pero con andamiaje oficial.

> **LangGraph:** librería para construir sistemas multiagente como grafos dirigidos con estado compartido y explícito.

Lo que ya sabés en Python puro tiene su equivalente directo en LangGraph:

| Concepto Python puro (E00–E06) | Equivalente en LangGraph |
|---|---|
| `dict` pasado entre funciones | `TypedDict` como `State` |
| Función `fn(query) -> resultado` | Nodo: `fn(state) -> dict` |
| `if intent == 'hr': hr_agent()` | `add_conditional_edges` |
| Llamada secuencial manual | `add_edge(A, B)` |
| `handle_query()` que orquesta todo | `graph.compile()` + `app.invoke()` |


## Paso 1 — Elegí tu proveedor de LLM

Cambiá `PROVIDER` al proveedor que uses y ejecutá la celda. Te va a pedir la API key de forma segura con `getpass`.

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

## Paso 2 — Instalar LangGraph e imports

In [ ]:
!pip install langgraph -q

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph listo.")

## Sección 1 — Las cinco piezas de un grafo

> **StateGraph:** grafo dirigido donde cada nodo recibe el estado completo, modifica algunos campos, y lo pasa al siguiente nodo.

Las cinco piezas son siempre las mismas, en este orden:

| Pieza | Qué es | Código |
|---|---|---|
| **State** | TypedDict con todos los campos del sistema | `class State(TypedDict): ...` |
| **Nodos** | Funciones `fn(state) -> dict` que modifican campos | `def mi_nodo(state): return {"campo": ...}` |
| **Edges** | Conexiones fijas entre nodos | `graph.add_edge(A, B)` |
| **START / END** | Nodos especiales de entrada y salida | `graph.add_edge(START, primer_nodo)` |
| **compile + invoke** | Valida el grafo y lo ejecuta | `app = graph.compile(); app.invoke(estado)` |

El flujo del ejercicio:

```
START --> greet --> format_output --> END
          (LLM)         ↑
                   implementado
```

- `greet` usa el LLM para generar un saludo personalizado.
- `format_output` toma ese saludo y lo rodea con `>> ... <<`.


### El State

Cada campo del `TypedDict` representa un dato que el grafo produce o consume. Un nodo solo devuelve los campos que modifica; LangGraph fusiona ese dict parcial con el estado existente.

```python
# Correcto: devuelve solo el campo que este nodo produce
def mi_nodo(state: MyState) -> dict:
    return {"campo_nuevo": "valor"}

# Incorrecto: no hace falta devolver todo el state
def mi_nodo(state: MyState) -> MyState:
    state["campo_nuevo"] = "valor"
    return state
```

In [ ]:
class GreetState(TypedDict):
    name: str        # entrada: nombre de la persona a saludar
    message: str     # producido por greet
    formatted: str   # producido por format_output

print("State definido:", list(GreetState.__annotations__.keys()))

## Sección 2 — Los nodos

> **Nodo:** función Python que recibe el estado completo y devuelve un `dict` con solo los campos que modifica.

El nodo `greet` llama al LLM con `llm.invoke(prompt)`. Devuelve un `AIMessage`; el texto está en `.content`.

El nodo `format_output` rodea el mensaje con `>> ... <<`.

In [ ]:
def greet(state: GreetState) -> dict:
    response = llm.invoke(
        f"En una sola oración amigable y en español, saludá a {state['name']}. Respondé solo el saludo."
    )
    return {"message": response.content.strip()}


def format_output(state: GreetState) -> dict:
    return {"formatted": f">> {state['message']} <<"}

## Sección 3 — Construir, compilar e invocar

> **compile():** valida que el grafo sea correcto y devuelve un objeto ejecutable.

```
1. StateGraph(MiState)       → crear el contenedor
2. add_node("nombre", fn)    → registrar cada nodo
3. add_edge(origen, destino) → conectar los nodos
4. compile()                 → bloquear y validar
5. invoke(estado_inicial)    → ejecutar
```

Los tres edges conectan `START → greet → format_output → END`.

In [ ]:
graph = StateGraph(GreetState)

graph.add_node("greet",         greet)
graph.add_node("format_output", format_output)

graph.add_edge(START, "greet")
graph.add_edge("greet", "format_output")
graph.add_edge("format_output", END)

app = graph.compile()
print("Grafo compilado.")

`invoke` recibe el estado inicial y devuelve el estado final. El mensaje viene del LLM real.

In [ ]:
result = app.invoke({"name": "estudiante", "message": "", "formatted": ""})
print("Mensaje LLM:", result["message"])
print("Formateado: ", result["formatted"])

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    r = app.invoke({"name": "Ana", "message": "", "formatted": ""})
    assert isinstance(r["message"], str) and len(r["message"]) > 5, f"mensaje vacío: {r['message']}"
    assert r["formatted"].startswith(">>"), f"formatted debe empezar con >>: {r['formatted']}"
    assert r["formatted"].endswith("<<"), f"formatted debe terminar con <<: {r['formatted']}"
    assert r["message"] in r["formatted"],  f"mensaje debe estar dentro de formatted: {r['formatted']}"
    print("Checks E07 OK")

run_checks()

## ¿Qué aprendiste hoy?

- El `State` es el contrato explícito del grafo: todos los nodos lo leen y pueden modificarlo.
- Un nodo devuelve solo los campos que cambia; LangGraph hace el merge automáticamente.
- `llm.invoke(prompt)` dentro de un nodo es el punto de integración con el LLM real.
- `compile()` valida el grafo antes de ejecutarlo.

## Próximo ejercicio

En **E08** el LLM clasifica la intención del usuario y el grafo decide a qué nodo especialista ir.
